In [1]:
%load_ext autoreload
%autoreload 2

Imports

In [2]:
import pandas as pd

Read Raw Data

In [3]:
member_detail_df_raw = pd.read_csv("../data/raw/member_detail_sample.csv", dtype={"tax_ref_no_payroll": str,"natlidno": str,"recent_phone": str, "pr_mbr_no": str})
transactions_df_raw = pd.read_csv("../data/raw/transactions_sample.csv")
tp_investment_df_raw = pd.read_csv("../data/raw/tp_investment_sample.csv")

Validation

In [4]:
from validation import validate_all

validate_all(member_detail_df_raw, transactions_df_raw, tp_investment_df_raw)

Read Validated Data

In [5]:
member_detail_df = pd.read_csv('../data/primary/member_detail_validated.csv', dtype={"tax_ref_no_payroll": str,"natlidno": str,"recent_phone": str, "pr_mbr_no": str})
transactions_df = pd.read_csv('../data/primary/transactions_validated.csv')
tp_investment_df = pd.read_csv("../data/primary/two_pot_investment_validated.csv")
tp_investment_df['rbal_dt'] = pd.to_datetime(tp_investment_df['latest_rbal_dt']) - pd.Timedelta(days=1)

Generate Statement for each Member

In [6]:
from pathlib import Path

from PIL import Image
from datetime import datetime
from statement_templates.saccawu import generate
from dateutil.relativedelta import relativedelta

old_mutual_logo = Image.open("../assets/old_mutual_header.png")
saccawu_logo = Image.open("../assets/saccawu_logo.png")

max_transactions_date = pd.to_datetime(transactions_df["contribution_dt"]).max()
max_tp_investment_date = pd.to_datetime(tp_investment_df["rbal_dt"]).max()

min_max_date = min(max_transactions_date, max_tp_investment_date)

reporting_dt = min_max_date
# reporting_dt = datetime(2025, 12, 31)
# start_dt = reporting_dt - relativedelta(months=12)
start_dt = reporting_dt.replace(month=1, day=1)

# Filter transactions and two pot investments for period
transactions_df["contribution_dt"] = pd.to_datetime(transactions_df["contribution_dt"])
transactions_df_filtered = transactions_df[transactions_df["contribution_dt"].between(start_dt, reporting_dt)]

tp_investment_df["rbal_dt"] = pd.to_datetime(tp_investment_df["rbal_dt"])
tp_investment_df_filtered = tp_investment_df[tp_investment_df["rbal_dt"].between(start_dt, reporting_dt)]

# Loop through member detail to process each members statement
for _ , row in member_detail_df.iterrows():
    
    case_mbr_key = row['case_mbr_key']
    
    member_data = member_detail_df[member_detail_df['case_mbr_key'] == case_mbr_key]

    transactions_data = transactions_df_filtered[transactions_df_filtered['case_mbr_key'] == case_mbr_key]
    
    tp_investment_data = tp_investment_df_filtered[tp_investment_df_filtered['case_mbr_key'] == case_mbr_key].copy()

    output_path = Path(f"../statements/{case_mbr_key}_member_statement.pdf")

    generate.generate_statement(member_data, transactions_data, tp_investment_data, output_path, old_mutual_logo, saccawu_logo, reporting_dt, start_dt)